In this notebook, I extend the modeling work from the previous notebooks to the day-ahead load forecasting problem. Previously, I forecasted load for a given hour using all available information up to that hour, including the actual temperature for the forecasted hour. However, such high-resolution data is not always available in practice. Day-ahead forecasting introduces additional constraints. First, recent load data are not available; the latest load observations are at least 24 hours old and may be even older, depending on the hour of prediction and the day-ahead market closing time. Second, actual temperature data for the prediction hour are unavailable, and models must rely on weather forecasts, which introduces an additional source of error.

To simulate these conditions, I enforce the following constraints on the test data. Only load lags greater than 24 hours are used as features, and actual temperature is replaced with forecasted temperature. Forecasted temperature is simulated by adding random noise to the actual temperature while incorporating autocorrelation with the forecasted temperature from the previous timestep. To reflect the increasing uncertainty of longer-horizon forecasts, the standard deviation of the noise is scaled linearly with the hour of day, reaching its maximum at hour 24.

This notebook contains the dataset construction and weather forecasting code for the models. This ntoebook also contains the hyperparameter optimzation for the family of models I consider, namely Linear Regression, Ridge Regression, Lasso Regression, XGBoost, and Linear Regression with XGBoost modelled residuals.

In [288]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error

import pandas as pd
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_squared_error

from itertools import product


In [290]:
df = pd.read_csv("../data/processed/day_ahead_train.csv")

In [292]:
def simulate_long_forecast(actual_temp_series, base_std = 1.0, max_std = 5.0, correlation_factor = 0.5):
    
    simulated_forecast = []
    last_forecast = actual_temp_series.iloc[0]
    
    hours_in_day = 24
    
    for i, actual in enumerate(actual_temp_series):
        hour_of_day = i % hours_in_day
        
        # Scale uncertainty linearly within the day
        hour_std = base_std + (max_std - base_std) * (hour_of_day / (hours_in_day - 1))
        
        # Random error for this hour
        error = np.random.normal(loc = 0, scale = hour_std)
        
        # Forecast combines correlation with last forecast and random error
        new_forecast = last_forecast + correlation_factor * (actual - last_forecast) + error
        
        simulated_forecast.append(new_forecast)
        last_forecast = new_forecast
    
    return pd.Series(simulated_forecast, index = actual_temp_series.index, name = "temp_simulated")

In [55]:
for i in range(10):
    simulated_temp = simulate_long_forecast(df['avg_region_temp']).rename(f"forecast_{i + 1}")
    df = pd.concat([df, simulated_temp], axis = 1)

In [57]:
base_t = 60

for i in range(10):
    df[f"temp_6h_{i + 1}"] = df[f"forecast_{i + 1}"].rolling(6).mean()
    df[f"CDH_{i + 1}"] = (df[f"forecast_{i + 1}"] - base_t).clip(lower=0)
    df[f"HDH_{i + 1}"] = (base_t - df[f"forecast_{i + 1}"]).clip(lower=0) 

df["Load_lag_24h"] = df["Load"].shift(24)
df["Load_lag_48h"] = df["Load"].shift(48)
df["temp_lag_24h"] = df["avg_region_temp"].shift(24)
df = df.dropna()

In [59]:
df

,timestamp,Load,avg_region_temp,forecast_1,forecast_2,forecast_3,forecast_4,forecast_5,forecast_6,forecast_7,...,HDH_8,temp_6h_9,CDH_9,HDH_9,temp_6h_10,CDH_10,HDH_10,Load_lag_24h,Load_lag_48h,temp_lag_24h
6936,2022-10-17 00:00:00,1944,61.880,63.733351,65.604423,64.825909,64.519887,63.137447,67.215771,66.893236,...,0.000000,67.303408,4.371633,0.000000,63.157748,0.000000,1.870279,1980.0,2010.0,63.644
6937,2022-10-17 01:00:00,1894,61.088,62.157563,63.469537,62.084116,61.560210,61.824192,65.242636,63.037355,...,0.000000,65.608777,0.595788,0.000000,62.438415,0.551266,0.000000,1918.0,1942.0,63.572
6938,2022-10-17 02:00:00,1900,60.836,59.692483,61.059373,60.335271,61.560557,62.887374,63.450569,59.579688,...,0.000000,63.340444,0.000000,0.525894,60.538674,0.000000,1.317132,1891.0,1915.0,63.572
6939,2022-10-17 03:00:00,1884,60.692,57.441216,60.010477,61.855902,59.661396,62.143287,62.337104,61.449021,...,0.000000,63.478431,2.089449,0.000000,60.058463,0.540253,0.000000,1860.0,1925.0,63.464
6940,2022-10-17 04:00:00,2004,60.908,58.339865,60.778771,63.338300,61.700214,59.762588,61.331039,61.062843,...,0.000000,62.665716,0.368749,0.000000,58.264090,0.000000,1.880317,1859.0,1929.0,63.932
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8755,2022-12-31 19:00:00,2434,57.344,53.449449,58.345988,65.478509,62.001984,55.960443,59.749568,56.228505,...,1.162837,59.649564,1.803685,0.000000,57.478225,0.000000,5.483779,2505.0,2556.0,57.056
8756,2022-12-31 20:00:00,2343,57.596,50.674315,53.373343,59.687233,62.070409,63.144591,63.043801,41.636648,...,0.000000,60.132111,1.487658,0.000000,56.789094,0.000000,4.237876,2429.0,2469.0,57.020
8757,2022-12-31 21:00:00,2253,57.596,52.536564,58.123262,59.201525,58.721409,56.354312,60.974494,52.412662,...,0.000000,59.857380,2.478883,0.000000,56.329414,0.000000,4.382497,2332.0,2346.0,56.444
8758,2022-12-31 22:00:00,2159,57.272,53.178901,66.821365,59.371251,57.387179,62.765726,62.077685,56.653463,...,0.534470,58.630876,0.000000,2.069485,56.422488,0.000000,3.663824,2191.0,2189.0,56.516


In [63]:
df.columns

Index(['Year', 'Month', 'Day', 'Hour', 'Load', 'Site-1 Temp', 'Site-2 Temp',
       'Site-3 Temp', 'Site-4 Temp', 'Site-5 Temp', 'Site-1 GHI', 'Site-2 GHI',
       'Site-3 GHI', 'Site-4 GHI', 'Site-5 GHI', 'temp_actual',
       'temp_forecast_1', 'temp_forecast_2', 'temp_forecast_3',
       'temp_forecast_4', 'temp_forecast_5', 'temp_forecast_6',
       'temp_forecast_7', 'temp_forecast_8', 'temp_forecast_9',
       'temp_forecast_10', 'timestamp', 'Hour_sin', 'Hour_cos',
       'temp_6h_actual', 'CDH_actual', 'HDH_actual', 'temp_6h_forecast_1',
       'CDH_forecast_1', 'HDH_forecast_1', 'temp_6h_forecast_2',
       'CDH_forecast_2', 'HDH_forecast_2', 'temp_6h_forecast_3',
       'CDH_forecast_3', 'HDH_forecast_3', 'temp_6h_forecast_4',
       'CDH_forecast_4', 'HDH_forecast_4', 'temp_6h_forecast_5',
       'CDH_forecast_5', 'HDH_forecast_5', 'temp_6h_forecast_6',
       'CDH_forecast_6', 'HDH_forecast_6', 'temp_6h_forecast_7',
       'CDH_forecast_7', 'HDH_forecast_7', 'temp_6h_foreca

In [65]:
splits_df_loc = "../data/splits/split_bounds.csv"
splits_df = pd.read_csv(splits_df_loc)

def linear_rmse_on_features(df, features_to_train, do_print = False):

    if do_print:
        print("Now training on: ", features_to_train)

    rmse_list = []

    for i in range(1, 9):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask]
        val_split = df[val_mask]
    
        X_train = train_split[features_to_train]
        y_train = train_split["Load"]
    
        X_val = val_split[features_to_train]
        y_val = val_split["Load"]
    
        model = LinearRegression()
        model.fit(X_train, y_train)
    
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmse_list.append(rmse)
    if do_print:
        print("RMSEs: ", rmse_list)
        print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")
    return np.mean(rmse_list)

def linear_reg_rmse_on_features(df, features_to_train, regression_type = "Lasso", alpha = 0.1, do_print = False):

    if do_print:
        print("Now training on: ", features_to_train)

    rmse_list = []

    for i in range(1, 9):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask]
        val_split = df[val_mask]
    
        X_train = train_split[features_to_train]
        y_train = train_split["Load"]
    
        X_val = val_split[features_to_train]
        y_val = val_split["Load"]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_val_scaled = scaler.transform(X_val)

        if regression_type == "Lasso":
            model = Lasso(alpha = alpha, max_iter=20_000)

        if regression_type == "Ridge":
            model = Ridge(alpha = alpha, max_iter=20_000)
            
        model.fit(X_train_scaled, y_train)
    
        y_pred = model.predict(X_val_scaled)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmse_list.append(rmse)
    if do_print:
        print("RMSEs: ", rmse_list)
        print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")
    return np.mean(rmse_list)


In [83]:
features_to_train = ["temp_actual", "temp_6h_actual", "CDH_actual", "HDH_actual", "temp_actual_lag_24h", "Load_lag_24h", "Load_lag_48h", "is_weekend", "is_notable_day"]

print("-"*50)
print(f"Linear Regression")
print("-"*50)

hour_errors = []

for i in range(24):
    
    hour_df = df[df["Hour"] == i]
    hour_errors.append(linear_rmse_on_features(hour_df, features_to_train))

print(f"Max error at hour {np.argmax(hour_errors)}: {np.max(hour_errors)}\n\n")

print("-"*50)
print(f"Lasso Regression")
print("-"*50)

alphas_lasso = np.logspace(-1, 1, 10)
alpha_max_errors = []

for alpha in alphas_lasso:
    print(f"Lasso alpha = {alpha}")
    hour_errors = []
    
    for i in range(24):
        
        hour_df = df[df["Hour"] == i]
        hour_errors.append(linear_reg_rmse_on_features(hour_df, features_to_train, regression_type = "Lasso", alpha = alpha))
        
    print(f"Max error at hour {np.argmax(hour_errors)}: {np.max(hour_errors)}\n\n")
    alpha_max_errors.append(np.max(hour_errors))

print(f"Best performance at worst hour: {np.min(alpha_max_errors)}, alpha = {alphas_lasso[np.argmin(alpha_max_errors)]}\n\n")
    
print("-"*50)
print(f"Ridge Regression")
print("-"*50)

alphas_ridge = np.logspace(-3, 3, 13)
alpha_max_errors = []

for alpha in alphas_ridge:
    print(f"Ridge alpha = {alpha}")
    hour_errors = []
    
    for i in range(24):
        
        hour_df = df[df["Hour"] == i]
        hour_errors.append(linear_reg_rmse_on_features(hour_df, features_to_train, regression_type = "Ridge", alpha = alpha))
        
    print(f"Max error at hour {np.argmax(hour_errors)}: {np.max(hour_errors)}\n\n")
    alpha_max_errors.append(np.max(hour_errors))

print(f"Best performance at worst hour: {np.min(alpha_max_errors)}, alpha = {alphas_ridge[np.argmin(alpha_max_errors)]}\n\n")


--------------------------------------------------
Linear Regression
--------------------------------------------------
Max error at hour 12: 253.06131510514382


--------------------------------------------------
Lasso Regression
--------------------------------------------------
Lasso alpha = 0.1
Max error at hour 12: 253.01632185242892


Lasso alpha = 0.16681005372000587
Max error at hour 12: 252.97600821148524


Lasso alpha = 0.2782559402207124
Max error at hour 12: 252.9126073498234


Lasso alpha = 0.46415888336127786
Max error at hour 12: 252.81177032770776


Lasso alpha = 0.774263682681127
Max error at hour 12: 252.65737722761992


Lasso alpha = 1.291549665014884
Max error at hour 12: 252.43992471688665


Lasso alpha = 2.1544346900318834
Max error at hour 12: 251.8241106439495


Lasso alpha = 3.593813663804626
Max error at hour 12: 251.36608583814268


Lasso alpha = 5.994842503189409
Max error at hour 12: 251.64960870395907


Lasso alpha = 10.0
Max error at hour 12: 252.78392174

In [85]:
def xgbr_rmse_on_features(df, features_to_train, xgbr_depth = 3, xgbr_estimators = 200, xgbr_lr = 0.1, xgbr_min_child_weight = 5, do_print = False):

    if do_print:
        print("Now training on: ", features_to_train)
    
    rmse_list = []

    for i in range(1, 9):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask].drop(columns=["timestamp"])
        val_split = df[val_mask].drop(columns=["timestamp"])
    
        X_train = df[train_mask][features_to_train]
        y_train = df[train_mask]["Load"]
    
        X_val = df[val_mask][features_to_train]
        y_val = df[val_mask]["Load"]
    
        model = XGBRegressor(objective='reg:squarederror', n_estimators = xgbr_estimators, learning_rate = xgbr_lr, max_depth = xgbr_depth, min_child_weight = xgbr_min_child_weight, subsample=0.8)

        model.fit(X_train, y_train)
    
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmse_list.append(rmse)
        
    if do_print:
        print("RMSEs: ", rmse_list)
        print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")
    return np.mean(rmse_list)

In [87]:
depths = [i for i in range(2, 5)]
estimators = [i for i in range(50, 300, 50)]
min_child_weight = [10, 20]
lr = [0.05, 0.1]


param_grid = {
    "depth": depths,
    "estimators": estimators,
    "min_child_weight": min_child_weight,
    "lr": lr
}

print("\nHour Depth Est  MCW  LR    RMSE")
print("-" * 40)

xgb_results = []

for hour in range(0, 24):
    for depth, n_est, mcw, lr_ in product(
        param_grid["depth"],
        param_grid["estimators"],
        param_grid["min_child_weight"],
        param_grid["lr"]
    ):
        rmse = xgbr_rmse_on_features(
            df=df[df["Hour"] == hour],
            features_to_train=features_to_train,
            xgbr_depth=depth,
            xgbr_estimators=n_est,
            xgbr_lr=lr_,
            xgbr_min_child_weight=mcw,
            do_print=False
        )
        print(
            f"{hour:02d}   "
            f"{depth:^5} "
            f"{n_est:>3}   "
            f"{mcw:>2}   "
            f"{lr_:>4.2f}  "
            f"{rmse:>8.4f}")
    
        xgb_results.append({
            "hour": hour,
            "max_depth": depth,
            "n_estimators": n_est,
            "min_child_weight": mcw,
            "learning_rate": lr_,
            "mean_rmse": rmse
        })



Hour Depth Est  MCW  LR    RMSE
----------------------------------------
00     2    50   10   0.05   92.9229
00     2    50   10   0.10   85.4627
00     2    50   20   0.05   92.4640
00     2    50   20   0.10   86.7975
00     2   100   10   0.05   85.6027
00     2   100   10   0.10   84.2690
00     2   100   20   0.05   86.3881
00     2   100   20   0.10   85.4843
00     2   150   10   0.05   84.1510
00     2   150   10   0.10   84.2264
00     2   150   20   0.05   85.7448
00     2   150   20   0.10   85.2633
00     2   200   10   0.05   83.9875
00     2   200   10   0.10   83.7975
00     2   200   20   0.05   85.3353
00     2   200   20   0.10   85.2964
00     2   250   10   0.05   83.8796
00     2   250   10   0.10   83.6221
00     2   250   20   0.05   85.4271
00     2   250   20   0.10   85.3829
00     3    50   10   0.05   91.0417
00     3    50   10   0.10   84.6575
00     3    50   20   0.05   90.3996
00     3    50   20   0.10   86.0190
00     3   100   10   0.05   84.8625
0

In [91]:
xgb_results_df = pd.DataFrame(xgb_results)

rmse_by_hour = xgb_results_df.pivot_table(
    index=["max_depth", "n_estimators", "min_child_weight", "learning_rate"],
    columns="hour",
    values="mean_rmse"
).reset_index()

best_per_hour_xgb = xgb_results_df.loc[xgb_results_df.groupby("hour")["mean_rmse"].idxmin()].reset_index(drop=True)

In [95]:
best_per_hour_xgb

,hour,max_depth,n_estimators,min_child_weight,learning_rate,mean_rmse
0,0,2,250,10,0.10,83.622075
1,1,3,100,10,0.10,86.268905
2,2,2,250,10,0.10,77.928198
3,3,2,250,20,0.05,74.881408
4,4,4,50,20,0.10,77.350257
5,5,4,50,10,0.10,93.449283
6,6,4,100,10,0.05,120.648700
7,7,3,200,20,0.05,154.592488
8,8,2,150,10,0.10,182.113663
9,9,4,100,10,0.05,206.955672


In [121]:
best_per_hour_hyperparams = best_per_hour_xgb[["max_depth", "n_estimators", "min_child_weight", "learning_rate"]]
best_at_max_hour_hyperparams = best_per_hour_xgb.loc[best_per_hour_xgb["mean_rmse"].idxmax()].to_frame().T

best_per_hour_hyperparams.to_csv("../src/models/best_per_hour_xgb_hyperparams.csv", index=False)
best_at_max_hour_hyperparams.to_csv("../src/models/best_max_hour_xgb_hyperparams.csv", index=False)

In [113]:
def linear_xgbr_rmse_on_features(df, features_to_train, xgbr_depth = 3, xgbr_estimators = 200, xgbr_lr = 0.1, xgbr_min_child_weight = 5, do_print = False):

    if do_print:
        print("Now training on: ", features_to_train)

    rmse_list = []
    df = df.sort_values(by=["timestamp"])

    for i in range(1, 9):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask].drop(columns=["timestamp", "Load"])
        val_split = df[val_mask].drop(columns=["timestamp", "Load"])
    
        X_train = train_split[features_to_train]
        y_train = df[train_mask]["Load"]
    
        X_val = val_split[features_to_train]
        y_val = df[val_mask]["Load"]

        n_train_samples = len(X_train)
        initial_len = int(0.6 * n_train_samples)
        step_size = 24

        oos_residuals = np.full(n_train_samples, np.nan)

        for start in range(initial_len, n_train_samples, step_size):
            end = min(start + step_size, n_train_samples)

            wf_lin = LinearRegression()
            wf_lin.fit(X_train.iloc[:start], y_train.iloc[:start])
        
            preds = wf_lin.predict(X_train.iloc[start:end])

            oos_residuals[start:end] = y_train.iloc[start:end] - preds

        mask = ~np.isnan(oos_residuals)

        xgbr = XGBRegressor(objective='reg:squarederror', n_estimators = xgbr_estimators, learning_rate=xgbr_lr, max_depth = xgbr_depth, min_child_weight = xgbr_min_child_weight, subsample=0.8, random_state=12)
        xgbr.fit(train_split.iloc[mask], oos_residuals[mask])

        model = LinearRegression()
        model.fit(X_train, y_train)
    
        y_lin_pred = model.predict(X_val)
        y_xgbr_pred = xgbr.predict(val_split)
        rmse = root_mean_squared_error(y_val, y_lin_pred + y_xgbr_pred)
    
        rmse_list.append(rmse)

    if do_print:
        print("RMSEs: ", rmse_list)
        print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")
        
    return np.mean(rmse_list)

In [119]:
features_to_train = ["temp_actual", "temp_6h_actual", "CDH_actual", "HDH_actual", "temp_actual_lag_24h", "Load_lag_24h", "Load_lag_48h", "is_weekend", "is_notable_day"]


depths = [i for i in range(2, 5)]
estimators = [i for i in range(50, 300, 50)]
min_child_weight = [10, 20]
lr = [0.05, 0.1]


param_grid = {
    "depth": depths,
    "estimators": estimators,
    "min_child_weight": min_child_weight,
    "lr": lr
}

print("\nHour Depth Est  MCW  LR    RMSE")
print("-" * 40)

lin_xgb_results = []

for hour in range(0, 24):
    for depth, n_est, mcw, lr_ in product(
        param_grid["depth"],
        param_grid["estimators"],
        param_grid["min_child_weight"],
        param_grid["lr"]
    ):

        rmse = linear_xgbr_rmse_on_features(
            df=df[df["Hour"] == hour],
            features_to_train=features_to_train,
            xgbr_depth=depth,
            xgbr_estimators=n_est,
            xgbr_lr=lr_,
            xgbr_min_child_weight=mcw,
            do_print=False
        )
        print(
            f"{hour:02d}   "
            f"{depth:^5} "
            f"{n_est:>3}   "
            f"{mcw:>2}   "
            f"{lr_:>4.2f}  "
            f"{rmse:>8.4f}")
    
        lin_xgb_results.append({
            "hour": hour,
            "max_depth": depth,
            "n_estimators": n_est,
            "min_child_weight": mcw,
            "learning_rate": lr_,
            "mean_rmse": rmse
        })



Hour Depth Est  MCW  LR    RMSE
----------------------------------------
00     2    50   10   0.05   74.3290
00     2    50   10   0.10   75.3166
00     2    50   20   0.05   75.4180
00     2    50   20   0.10   76.4439
00     2   100   10   0.05   75.2162
00     2   100   10   0.10   76.7734
00     2   100   20   0.05   76.3464
00     2   100   20   0.10   78.0928
00     2   150   10   0.05   76.3183
00     2   150   10   0.10   78.7754
00     2   150   20   0.05   76.9917
00     2   150   20   0.10   79.6367
00     2   200   10   0.05   77.3581
00     2   200   10   0.10   80.0924
00     2   200   20   0.05   77.5587
00     2   200   20   0.10   80.4906
00     2   250   10   0.05   77.8115
00     2   250   10   0.10   80.3573
00     2   250   20   0.05   78.1361
00     2   250   20   0.10   80.8711
00     3    50   10   0.05   74.4392
00     3    50   10   0.10   75.7957
00     3    50   20   0.05   75.6153
00     3    50   20   0.10   76.0583
00     3   100   10   0.05   76.0517
0

In [127]:
lin_xgb_results_df = pd.DataFrame(lin_xgb_results)

rmse_by_hour = lin_xgb_results_df.pivot_table(
    index=["max_depth", "n_estimators", "min_child_weight", "learning_rate"],
    columns="hour",
    values="mean_rmse"
).reset_index()

best_per_hour_lin_xgb = lin_xgb_results_df.loc[lin_xgb_results_df.groupby("hour")["mean_rmse"].idxmin()].reset_index(drop=True)

In [128]:
best_per_hour_lin_xgb

,hour,max_depth,n_estimators,min_child_weight,learning_rate,mean_rmse
0,0,4,50,10,0.05,74.239318
1,1,2,50,10,0.05,73.079033
2,2,2,50,10,0.05,67.754544
3,3,4,50,20,0.05,62.544141
4,4,2,50,20,0.05,62.222497
5,5,3,50,10,0.05,75.963719
6,6,4,50,20,0.05,102.400875
7,7,2,50,20,0.05,137.349189
8,8,4,100,10,0.05,168.636245
9,9,2,200,10,0.05,200.513561


In [131]:
best_per_hour_hyperparams = best_per_hour_lin_xgb[["max_depth", "n_estimators", "min_child_weight", "learning_rate"]]
best_at_max_hour_hyperparams = best_per_hour_lin_xgb.loc[best_per_hour_lin_xgb["mean_rmse"].idxmax()].to_frame().T

best_per_hour_hyperparams.to_csv("../src/models/best_per_hour_lin_xgb_hyperparams.csv", index=False)
best_at_max_hour_hyperparams.to_csv("../src/models/best_max_hour_lin_xgb_hyperparams.csv", index=False)